In [1]:
import numpy as np
import pandas as pd
from survivors.external import ClassifWrapSA, RegrWrapSA, SAWrapSA
from wrapSA import PiecewiseClassifWrapSA

In [2]:
from sklearn.base import clone

class PiecewiseCensorAwareClassifWrapSA(PiecewiseClassifWrapSA):
    """Piecewise-wrapper for discrete-time survival.

    Unlike the basic Piecewise wrapper, this version does not mark as class 0
    observations censored inside the current interval. Such rows are excluded
    from this interval's classifier, but still contribute to earlier intervals.
    """
    def __init__(self, model, times: int = 32, start_at_zero: bool = True):
        super().__init__(model=model, times=times, start_at_zero=start_at_zero)
        self.__name__ = f"PiecewiseCensorAwareClassifWrapSA({model.__class__.__name__}, times={times})"
        self.interval_stats_ = []

    def fit(self, X, y, time_col: str = "time", event_col: str = "cens"):
        t, e = self.timeWrap(y, time_col, event_col)
        left = float(np.min(t))
        right = float(np.max(t))
        if not np.isfinite(left) or not np.isfinite(right) or left >= right:
            left = 0.0
            right = 1.0
        self._bounds = (left, right)

        self.bin_edges_ = self._build_piecewise_bin_edges(t, self.times, self.start_at_zero)
        self.models_ = []
        self.interval_stats_ = []

        for j in range(1, len(self.bin_edges_)):
            a = self.bin_edges_[j - 1]
            b = self.bin_edges_[j]

            at_risk = t > a
            positive = (e == 1) & (t > a) & (t <= b)
            negative = t > b
            usable = at_risk & (positive | negative)
            yj = positive[usable].astype(int)

            if hasattr(X, "iloc"):
                Xj = X.iloc[usable]
            else:
                Xj = X[usable]

            self.interval_stats_.append({
                "left": float(a),
                "right": float(b),
                "at_risk": int(np.sum(at_risk)),
                "used": int(np.sum(usable)),
                "events": int(np.sum(positive[usable])),
                "survived_past_interval": int(np.sum(negative[usable])),
                "censored_inside_interval": int(np.sum(at_risk & (~positive) & (~negative))),
            })

            if len(yj) == 0:
                self.models_.append(None)
                continue

            if np.all(yj == 0):
                self.models_.append(0.0)
                continue

            if np.all(yj == 1):
                self.models_.append(1.0)
                continue

            m = clone(self.model)
            m.fit(Xj, yj)
            self.models_.append(m)

        return self

In [3]:
from sklearn.metrics import root_mean_squared_error, r2_score
from scipy.stats import spearmanr
# РњРµС‚СЂРёРєРё СЂРµРіСЂРµСЃСЃРёРё
# - RMSE (Root Mean Squared Error)
# - R^2 (Coefficient of Determination)
# - MAPE (Mean Absolute Percentage Error)
# - MEDAPE (Median Absolute Percentage Error)
# - Spearman РєРѕСЂСЂРµР»СЏС†РёСЏ
# - RMSLE (Root Mean Squared Logarithmic Error)

rmse_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: root_mean_squared_error(y_tst["time"], pred_time)
r2_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: r2_score(y_tst["time"], pred_time)
mape_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: np.mean(np.abs((y_tst["time"] - pred_time) / np.maximum(y_tst["time"], 1))) * 100
medape_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: np.median(np.abs((y_tst["time"] - pred_time) / np.maximum(y_tst["time"], 1))) * 100
spearman_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: spearmanr(y_tst["time"], pred_time)[0]
rmsle_exp_time = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: np.sqrt(np.mean((np.log1p(np.clip(y_tst["time"], 0, None)) - np.log1p(np.clip(pred_time, 0, None)))**2))

In [4]:
from sklearn.metrics import root_mean_squared_error, roc_auc_score, log_loss
# РњРµС‚СЂРёРєРё РєР»Р°СЃСЃРёС„РёРєР°С†РёРё
# - AUC РІРµСЂРѕСЏС‚РЅРѕСЃС‚Рё СЃРѕР±С‹С‚РёСЏ
# - log-loss РІРµСЂРѕСЏС‚РЅРѕСЃС‚Рё СЃРѕР±С‹С‚РёСЏ
# - RMSE РёСЃС…РѕРґР°

def find_sf_at_truetime(pred_sf, event_time, bins):
    idx_pred = np.clip(np.searchsorted(bins, event_time), 0, len(bins) - 1)
    proba = np.take_along_axis(pred_sf, idx_pred[:, np.newaxis], axis=1).squeeze()
    return proba

## example
# true_times = np.array([1, 19, 21, 31])
# bins = np.array([10,20,30])
# sf = np.array([[0.9, 0.8, 0.7], 
#                [0.7, 0.6, 0.5], 
#                [0.5, 0.4, 0.3],
#                [0.05, 0.04, 0.03]])
# print(np.mean(sf, axis=1))
# print(find_sf_at_truetime(sf, true_times, bins))  # [0.9  0.6  0.3  0.03] 

auc_event = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: roc_auc_score(y_tst["cens"].astype(int), 1 - np.mean(pred_sf, axis=1))
log_loss_event = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: log_loss(y_tst["cens"], 1 - np.mean(pred_sf, axis=1))
rmse_event = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: root_mean_squared_error(y_tst["cens"], 1 - np.mean(pred_sf, axis=1))

# auc_event_T = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: roc_auc_score(y_tst["cens"].astype(int), 1 - find_sf_at_truetime(pred_sf, y_tst["time"], bins))
# log_loss_event_T = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: log_loss(y_tst["cens"], 1 - find_sf_at_truetime(pred_sf, y_tst["time"], bins))
# rmse_event_T = lambda y_tr, y_tst, pred_time, pred_sf, pred_hf, bins: root_mean_squared_error(y_tst["cens"], 1 - find_sf_at_truetime(pred_sf, y_tst["time"], bins))

In [5]:
from survivors.experiments import grid as exp
import survivors.datasets as ds

l_metrics = ["CI", "IBS", "AUPRC", "RMSE_TIME", "R2_TIME", 
             "AUC_EVENT", "LOGLOSS_EVENT", "RMSE_EVENT",
            #  "AUC_EVENT_T", "LOGLOSS_EVENT_T", "RMSE_EVENT_T",
             "MAPE_TIME", "MEDAPE_TIME", "SPEARMAN_TIME", "RMSLE_TIME"]
experim = exp.Experiments(folds=5, mode="CV+SAMPLE")
experim.add_new_metric("RMSE_TIME", rmse_exp_time)
experim.add_new_metric("R2_TIME", r2_exp_time)
experim.add_new_metric("MAPE_TIME", mape_exp_time)
experim.add_new_metric("MEDAPE_TIME", medape_exp_time)
experim.add_new_metric("SPEARMAN_TIME", spearman_exp_time)
experim.add_new_metric("RMSLE_TIME", rmsle_exp_time)
experim.add_new_metric("AUC_EVENT", auc_event)
experim.add_new_metric("LOGLOSS_EVENT", log_loss_event)
experim.add_new_metric("RMSE_EVENT", rmse_event)
experim.set_metrics(l_metrics)

In [6]:
# Р“РёРїРµСЂРїР°СЂР°РјРµС‚СЂС‹ Рё РјРѕРґРµР»Рё РєР»Р°СЃСЃРёС„РёРєР°С†РёРё

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# CLASS_PARAM_GRIDS = {
#     "logistic_regression": dict(),
#     "svc": dict(),
#     "knn_classifier": dict(),
#     "decision_tree_classifier": dict(),
#     "random_forest_classifier": dict(),
#     "gradient_boosting_classifier": dict()
# }

CLASS_PARAM_GRIDS = {
    "logistic_regression": {
        "penalty": ["l2"],
        "C": [0.01, 0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"],
        "class_weight": [None, "balanced"],
        "max_iter": [1000],
    },
    "svc": {
        "kernel": ["linear", "rbf"],
        "C": [0.1, 1, 10],
        "class_weight": [None, "balanced"],
        "probability": [True],
    },
    "knn_classifier": {
        "n_neighbors": [5, 10, 20],
        "weights": ["uniform", "distance"],
    },
    "decision_tree_classifier": {
        "max_depth": [5, 10, 20],
        "min_samples_split": [2, 10],
        "min_samples_leaf": [1, 5],
        "criterion": ["gini", "entropy"],
    },
    "random_forest_classifier": {
        "n_estimators": [100, 300],
        "max_depth": [10, 30],
        "min_samples_split": [2, 10],
        "min_samples_leaf": [1, 5],
    },
    "gradient_boosting_classifier": {
        "n_estimators": [100, 300],
        "learning_rate": [0.05, 0.1],
        "max_depth": [2, 3],
        "subsample": [0.7, 1.0],
    }
}

experim.add_method(ClassifWrapSA(LogisticRegression()), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(LogisticRegression(), times=4), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(LogisticRegression(), times=8), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(LogisticRegression(), times=16), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(LogisticRegression(), times=32), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseClassifWrapSA(LogisticRegression(), times=4), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseClassifWrapSA(LogisticRegression(), times=8), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseClassifWrapSA(LogisticRegression(), times=16), CLASS_PARAM_GRIDS['logistic_regression'])
experim.add_method(PiecewiseClassifWrapSA(LogisticRegression(), times=32), CLASS_PARAM_GRIDS['logistic_regression'])
#experim.add_method(ClassifWrapSA(SVC()), CLASS_PARAM_GRIDS['svc'])
experim.add_method(ClassifWrapSA(KNeighborsClassifier()), CLASS_PARAM_GRIDS['knn_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(KNeighborsClassifier(), times=4), CLASS_PARAM_GRIDS['knn_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(KNeighborsClassifier(), times=8), CLASS_PARAM_GRIDS['knn_classifier'])
#experim.add_method(PiecewiseCensorAwareClassifWrapSA(KNeighborsClassifier(), times=16), CLASS_PARAM_GRIDS['knn_classifier'])
#experim.add_method(PiecewiseCensorAwareClassifWrapSA(KNeighborsClassifier(), times=32), CLASS_PARAM_GRIDS['knn_classifier'])
experim.add_method(PiecewiseClassifWrapSA(KNeighborsClassifier(), times=4), CLASS_PARAM_GRIDS['knn_classifier'])
experim.add_method(PiecewiseClassifWrapSA(KNeighborsClassifier(), times=8), CLASS_PARAM_GRIDS['knn_classifier'])
#experim.add_method(PiecewiseClassifWrapSA(KNeighborsClassifier(), times=16), CLASS_PARAM_GRIDS['knn_classifier'])
#experim.add_method(PiecewiseClassifWrapSA(KNeighborsClassifier(), times=32), CLASS_PARAM_GRIDS['knn_classifier'])
experim.add_method(ClassifWrapSA(DecisionTreeClassifier()), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(DecisionTreeClassifier(), times=4), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(DecisionTreeClassifier(), times=8), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(DecisionTreeClassifier(), times=16), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(DecisionTreeClassifier(), times=32), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseClassifWrapSA(DecisionTreeClassifier(), times=4), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseClassifWrapSA(DecisionTreeClassifier(), times=8), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseClassifWrapSA(DecisionTreeClassifier(), times=16), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(PiecewiseClassifWrapSA(DecisionTreeClassifier(), times=32), CLASS_PARAM_GRIDS['decision_tree_classifier'])
experim.add_method(ClassifWrapSA(RandomForestClassifier()), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(RandomForestClassifier(), times=4), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(RandomForestClassifier(), times=8), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(RandomForestClassifier(), times=16), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(RandomForestClassifier(), times=32), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseClassifWrapSA(RandomForestClassifier(), times=4), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseClassifWrapSA(RandomForestClassifier(), times=8), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseClassifWrapSA(RandomForestClassifier(), times=16), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(PiecewiseClassifWrapSA(RandomForestClassifier(), times=32), CLASS_PARAM_GRIDS['random_forest_classifier'])
experim.add_method(ClassifWrapSA(GradientBoostingClassifier()), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(GradientBoostingClassifier(), times=4), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(GradientBoostingClassifier(), times=8), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(GradientBoostingClassifier(), times=16), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseCensorAwareClassifWrapSA(GradientBoostingClassifier(), times=32), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseClassifWrapSA(GradientBoostingClassifier(), times=4), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseClassifWrapSA(GradientBoostingClassifier(), times=8), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseClassifWrapSA(GradientBoostingClassifier(), times=16), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])
experim.add_method(PiecewiseClassifWrapSA(GradientBoostingClassifier(), times=32), CLASS_PARAM_GRIDS['gradient_boosting_classifier'])


In [7]:
# Р“РёРїРµСЂРїР°СЂР°РјРµС‚СЂС‹ Рё РјРѕРґРµР»Рё СЂРµРіСЂРµСЃСЃРёРё

from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

REGR_PARAM_GRIDS = {
    "elastic_net": dict(),
    "decision_tree_regressor": dict(),
    "random_forest_regressor": dict(),
    "gradient_boosting_regressor": dict(),
    "svr": dict(),
    "knn_regressor": dict()
}

# REGR_PARAM_GRIDS = {
#     "elastic_net": {
#         "alpha": [0.001, 0.01, 0.1],
#         "l1_ratio": [0.2, 0.5, 0.8],
#         "max_iter": [1000, 5000],
#     }, 
#     "decision_tree_regressor": {
#         "max_depth": [5, 10, 20],
#         "min_samples_split": [2, 10],
#         "min_samples_leaf": [1, 5],
#         "criterion": ["squared_error", "friedman_mse"],
#     },
#     "random_forest_regressor": {
#         "n_estimators": [100, 300],
#         "max_depth": [10, 30],
#         "min_samples_split": [2, 10],
#         "min_samples_leaf": [1, 5],
#     },
#     "gradient_boosting_regressor": {
#         "n_estimators": [100, 300],
#         "learning_rate": [0.05, 0.1],
#         "max_depth": [2, 3],
#         "subsample": [0.7, 1.0],
#     },
#     "svr": {
#         "kernel": ["linear", "rbf"],
#         "C": [0.1, 1, 10],
#         "epsilon": [0.1, 0.2],
#     },
#     "knn_regressor": {
#         "n_neighbors": [5, 10, 20],
#         "weights": ["uniform", "distance"],
#     }
# }

# experim.add_method(RegrWrapSA(ElasticNet()), REGR_PARAM_GRIDS['elastic_net'])
# experim.add_method(RegrWrapSA(DecisionTreeRegressor()), REGR_PARAM_GRIDS['decision_tree_regressor'])
# experim.add_method(RegrWrapSA(RandomForestRegressor()), REGR_PARAM_GRIDS['random_forest_regressor'])
# experim.add_method(RegrWrapSA(GradientBoostingRegressor()), REGR_PARAM_GRIDS['gradient_boosting_regressor'])
# experim.add_method(RegrWrapSA(SVR()), REGR_PARAM_GRIDS['svr'])
# experim.add_method(RegrWrapSA(KNeighborsRegressor()), REGR_PARAM_GRIDS['knn_regressor'])

In [8]:
# Р“РёРїРµСЂРїР°СЂР°РјРµС‚СЂС‹ РјРѕРґРµР»РµР№ РІС‹Р¶РёРІР°РµРјРѕСЃС‚Рё (РІРЅРµС€РЅРёРµ РґР»СЏ survivors)

from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.tree import SurvivalTree
from sksurv.ensemble import RandomSurvivalForest
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from lifelines import KaplanMeierFitter

EXTERNAL_SURV_PARAM_GRIDS = {
    "km": dict(),
    "cox_ph": dict(),
    "random_survival_forest": dict(),
    "survival_tree": dict(),
    "gbsa": dict(),
}

# EXTERNAL_SURV_PARAM_GRIDS = {
#     "km": {},
#     "cox_ph": {
#         'alpha': [100, 10, 1, 0.1, 0.01, 0.001],
#         'ties': ["breslow"]
#     },
#     "random_survival_forest": {
#         'n_estimators': [50],
#         'max_depth': [5, 20],
#         'min_samples_leaf': [0.001, 0.01, 0.1, 0.25],
#         "random_state": [123]
#     },
#     "survival_tree": {
#         'max_depth': [None, 20, 30],
#         'min_samples_leaf': [1, 10, 20],
#         'max_features': [None, "sqrt"],
#         "random_state": [123]
#     },
#     "gbsa": {
#         'loss': ["coxph"],
#         'learning_rate': [0.01, 0.05, 0.1, 0.5],
#         'n_estimators': [50],
#         'min_samples_leaf': [1, 10, 50, 100],
#         'max_features': ["sqrt"],
#         "random_state": [123]
#     },
# }

# experim.add_method(SAWrapSA(KaplanMeierFitter()), EXTERNAL_SURV_PARAM_GRIDS['km'])
# experim.add_method(CoxPHSurvivalAnalysis, EXTERNAL_SURV_PARAM_GRIDS['cox_ph'])
# experim.add_method(RandomSurvivalForest, EXTERNAL_SURV_PARAM_GRIDS['random_survival_forest'])
# experim.add_method(SurvivalTree, EXTERNAL_SURV_PARAM_GRIDS['survival_tree'])
# experim.add_method(GradientBoostingSurvivalAnalysis, EXTERNAL_SURV_PARAM_GRIDS['gbsa'])

In [9]:
# Р“РёРїРµСЂРїР°СЂР°РјРµС‚СЂС‹ РјРѕРґРµР»РµР№ РІС‹Р¶РёРІР°РµРјРѕСЃС‚Рё (РІРЅСѓС‚СЂРё survivors)

from survivors.tree import CRAID
from survivors.ensemble import ParallelBootstrapCRAID

INTERNAL_SURV_PARAM_GRIDS = {
    "CRAID": {"depth": [10]},
    "ParallelBootstrapCRAID": {"depth": [10]}
}

# INTERNAL_SURV_PARAM_GRIDS = {
#     "CRAID": {
#         "depth": [10],
#         "criterion": ["wilcoxon", "logrank"],
#         "l_reg": [0, 0.01, 0.1],
#         "min_samples_leaf": [0.05, 0.01, 0.001],
#         "categ": [categ]
#     },
#     "ParallelBootstrapCRAID": {
#         "n_estimators": [50],
#         "depth": [7],
#         "size_sample": [0.3, 0.7],
#         "l_reg": [0, 0.01, 0.1],
#         "criterion": ["tarone-ware", "wilcoxon"],
#         "min_samples_leaf": [0.05, 0.01],
#         "ens_metric_name": ["IBS_REMAIN"],
#         "max_features": ["sqrt"],
#         "categ": [categ]
#     }
# }

# experim.add_method(CRAID, INTERNAL_SURV_PARAM_GRIDS["CRAID"])
# experim.add_method(ParallelBootstrapCRAID, INTERNAL_SURV_PARAM_GRIDS["ParallelBootstrapCRAID"])

In [10]:
import os
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from copy import deepcopy
from multiprocessing import get_context
from pathlib import Path

from survivors.datasets import other as ds_other

warnings.filterwarnings("ignore")

DATASET_ALIASES = {}
REQUESTED_DATASETS = ["actg", "gbsg", "rott2", "smarto", "framingham", "support2"]

def canonical_dataset_name(name):
    return DATASET_ALIASES.get(name, name)

def resolve_loader(*names):
    for module in (ds, ds_other):
        for name in names:
            if hasattr(module, name):
                return getattr(module, name)
    raise AttributeError(f"Loader not found. Tried: {names}")

TARGET_DATASETS = list(dict.fromkeys(canonical_dataset_name(name) for name in REQUESTED_DATASETS))
DATASET_LOADERS = {
    "actg": resolve_loader("load_actg_dataset"),
    "gbsg": resolve_loader("load_gbsg_dataset"),
    "rott2": resolve_loader("load_rott2_dataset"),
    "smarto": resolve_loader("load_smarto_dataset"),
    "framingham": resolve_loader("load_Framingham_dataset", "load_framingham_dataset"),
    "support2": resolve_loader("load_support2_dataset"),
    # "pbc": resolve_loader("load_pbc_dataset"),
}

OUT_DIR = Path("UI/tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def run_dataset(dataset_name):
    dataset_name = canonical_dataset_name(dataset_name)
    if dataset_name not in DATASET_LOADERS:
        raise KeyError(f"Unknown dataset: {dataset_name}")

    X, y, features, categ, _ = DATASET_LOADERS[dataset_name]()
    local_experim = deepcopy(experim)
    local_experim.run_effective(X, y, verbose=0, stratify_best=[])
    df_results = local_experim.get_best_by_mode()

    out_path = OUT_DIR / f"Piecewise_{dataset_name}.xlsx"
    df_results.to_excel(out_path, index=False)
    return {
        "dataset": dataset_name,
        "rows": int(len(df_results)),
        "output": str(out_path.resolve()),
    }

max_workers = min(len(TARGET_DATASETS), os.cpu_count() or 1)
run_summary = []

try:
    mp_context = get_context("fork")
except ValueError:
    mp_context = None

print(f"Running {len(TARGET_DATASETS)} datasets with {max_workers} workers")

if mp_context is None:
    print("fork context is unavailable, falling back to sequential run")
    for requested_name in TARGET_DATASETS:
        try:
            info = run_dataset(requested_name)
            run_summary.append({"dataset": info["dataset"], "status": "ok", "rows": info["rows"], "output": info["output"]})
            print(f"[saved] {info['dataset']} -> {info['output']}")
        except Exception as exc:
            run_summary.append({"dataset": canonical_dataset_name(requested_name), "status": "error", "rows": None, "output": repr(exc)})
            print(f"[error] {requested_name}: {exc!r}")
else:
    with ProcessPoolExecutor(max_workers=max_workers, mp_context=mp_context) as executor:
        futures = {executor.submit(run_dataset, requested_name): requested_name for requested_name in TARGET_DATASETS}
        for future in as_completed(futures):
            requested_name = futures[future]
            try:
                info = future.result()
                run_summary.append({"dataset": info["dataset"], "status": "ok", "rows": info["rows"], "output": info["output"]})
                print(f"[saved] {info['dataset']} -> {info['output']}")
            except Exception as exc:
                run_summary.append({"dataset": canonical_dataset_name(requested_name), "status": "error", "rows": None, "output": repr(exc)})
                print(f"[error] {requested_name}: {exc!r}")

run_summary = pd.DataFrame(run_summary).sort_values(["status", "dataset"]).reset_index(drop=True)
run_summary

Running 6 datasets with 6 workers
<survivors.external.mlwrap.ClassifWrapSA object at 0x10a4744d0><survivors.external.mlwrap.ClassifWrapSA object at 0x10a4744d0><survivors.external.mlwrap.ClassifWrapSA object at 0x10a4744d0><survivors.external.mlwrap.ClassifWrapSA object at 0x10a4744d0>    {'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs'], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}{'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs'], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}{'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs'], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}<survivors.external.mlwrap.ClassifWrapSA object at 0x10a4744d0>{'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs'], 'class_weight': [None, 'balanced'], 'max_iter': [1000]}


 
{'penalty': ['l2'], 'C': [0.01, 0.1, 1, 10], 'solver': ['liblinear', 'lbfgs'], 'class_weight': [None, 'bal

,dataset,status,rows,output
0,actg,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...
1,framingham,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...
2,gbsg,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...
3,rott2,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...
4,smarto,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...
5,support2,ok,41,/Users/dimonzhi/Documents/proga/SAWrap/UI/tabl...


In [11]:
# df_results.columns

In [12]:
# for m in ["AUC_EVENT", "LOGLOSS_EVENT", "RMSE_EVENT"]:
#     df_res_sort = df_results[["METHOD", f"{m}_mean", f"{m}_T_mean"]].round(3).sort_values(f"{m}_mean")
#     display(df_res_sort)

# for m in ["RMSE_TIME", "R2_TIME", "MAPE_TIME", "SPEARMAN_TIME", "RMSLE_TIME", "MEDAPE_TIME"]:
#     df_res_sort = df_results[["METHOD", f"{m}_mean"]].round(3).sort_values(f"{m}_mean")
#     display(df_res_sort)

In [13]:
# pd.Series(y["time"]).describe()